In [1]:
import pandas as pd
import datetime as dt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

In [2]:
input_ventas = "../data/processed/tabla_principal_ventas.csv"
input_clustering = "../data/processed/segmentacion_clientes_final.csv"

In [3]:
# Cargamos tus datos limpios
df_v = pd.read_csv(input_ventas)
df_clusters = pd.read_csv(input_clustering)

for col in ['created_at', 'shipped_at', 'delivered_at']:
    df_v[col] = pd.to_datetime(df_v[col])

In [4]:
# 1. Buscamos la fecha del último pedido registrado en toda la tienda
fecha_max = df_v['created_at'].max()
# 2. Restamos 90 días de inactividad
fecha_corte = fecha_max - pd.Timedelta(days=90)

In [5]:
# 3. Identificamos la última compra de cada usuario
ultimo_pedido = df_v.groupby('user_id')['created_at'].max().reset_index()
# 4. Target = 1 si su última compra es anterior al periodo de inactividad, Target = 0, sino lo es
ultimo_pedido['target'] = (ultimo_pedido['created_at'] < fecha_corte).astype(int)

In [6]:
pistas = df_v.groupby('user_id').agg({
    'revenue': 'mean',   # Ticket medio (Revenue)
    'order_id': 'nunique',  # Frecuencia
    'is_returned': 'sum',   # Conflictividad (Devoluciones)
    'es_recurrente': 'max'  # Historial de lealtad
}).reset_index()

# Unimos las pistas con la respuesta (target)
df_fuga = pd.merge(pistas, ultimo_pedido[['user_id', 'target']], on='user_id')

In [7]:
# Definimos X (pistas) e Y (objetivo)
features = ['revenue', 'order_id', 'is_returned', 'es_recurrente']
X = df_fuga[features]
Y = df_fuga['target']

# Dividimos: 70% para que el modelo aprenda, 30% para probarlo
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.3, random_state=42)

# Entrenamos el Bosque Aleatorio
modelo = RandomForestClassifier(n_estimators=100, random_state=42)
modelo.fit(X_train, Y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [8]:
y_pred = modelo.predict(X_test)
print(classification_report(Y_test, y_pred))

              precision    recall  f1-score   support

           0       0.30      0.21      0.25      6104
           1       0.76      0.84      0.79     17910

    accuracy                           0.68     24014
   macro avg       0.53      0.52      0.52     24014
weighted avg       0.64      0.68      0.66     24014



In [9]:
df_fuga['probabilidad_fuga'] = modelo.predict_proba(X)[:, 1]

In [10]:
# Nivel de riesgo
def clasificar_riesgo(prob):
    if prob > 0.8: return 'Crítico'
    elif prob > 0.6: return 'Alto'
    elif prob > 0.4: return 'Medio'
    else: return 'Bajo'

df_fuga['nivel_riesgo'] = df_fuga['probabilidad_fuga'].apply(clasificar_riesgo)

In [11]:
df_fuga = pd.merge(df_fuga, df_clusters[['user_id', 'segmento_nombre']], on='user_id')

def recomendar_accion(row):
    prob = row['probabilidad_fuga']
    segmento = row['segmento_nombre']

    # 1. NIVEL CRÍTICO: VIP/Recurrente con riesgo muy alto
    if segmento in ['VIP', 'Recurrentes'] and prob > 0.8:
        return "ALERTA: Llamada Urgente + Regalo VIP"
    
    # 2. NIVEL PREVENTIVO: VIP/Recurrente con riesgo moderado
    elif segmento in ['VIP', 'Recurrentes'] and 0.5 < prob <= 0.8:
        return "DESCUENTO: Cupón 20% 'Te echamos de menos'"

    # 3. NIVEL CRECIMIENTO: Clientes Nuevos con poco riesgo
    elif segmento == 'Nuevos / Esporádicos' and prob <= 0.5:
        return "CROSS-SELL: Recomendación productos tendencia"

    # 4. NIVEL MANTENIMIENTO: Inactivos, Problemáticos o bajo riesgo
    else:
        return "Fidelización estándar: Newsletter y novedades"

df_fuga['accion_marketing'] = df_fuga.apply(recomendar_accion, axis=1)

In [12]:
columnas_tfm = ['user_id', 'segmento_nombre', 'probabilidad_fuga', 'nivel_riesgo', 'accion_marketing']
df_fuga[columnas_tfm].to_csv("../data/processed/prediccion_fuga.csv", index=False)